# Prompt upsampler console

A long-running interface to the FLUX 3 Image Klein prompt upsampler (`flux3.pup.Upsampler`, a 9B VLM on vLLM). Load it
once, then edit the **Inputs** cell (prompt, reference images, boxes) and re-run **Upsample** as often as you like.

- no images: text-to-image caption
- images: editing caption, references are numbered in the order given (max 10)
- boxes: layout mode; each box is `{"id": ..., "desc": ..., "bbox": [y_min, x_min, y_max, x_max]}` in 0-1 (or
  `src_bbox` / `tgt_bbox` for moving things), see `examples/inputs/*_boxes.json`

## Pick a GPU

Nothing runs until `GPU` is set. The index is the one `nvidia-smi` prints; it is exported as `CUDA_VISIBLE_DEVICES`
*before* torch is imported, so inside the notebook the chosen card is always `cuda:0` (vLLM has no device argument and
only honours this variable). Restart the kernel to switch.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GPU = None  # <-- set me, e.g. GPU = 0

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert GPU is not None, "set GPU to one of the indices above before running anything else"
assert "torch" not in sys.modules, "torch is already imported: restart the kernel, then run this cell first"
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
DEVICE = "cuda"  # == cuda:0 == physical GPU above

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
os.environ.setdefault("WEIGHTS_DIR", str(REPO / "weights"))  # flux3 reads this at import time
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))


## Load the model

`MEMORY` is vLLM's share of the GPU. 0.85 when the upsampler is alone on the card; lower it (the pipeline uses 0.35) if a
DiT shares the GPU. The weights live at `WEIGHTS_DIR/pup`.

In [ ]:
from flux3.pup import MODEL, Upsampler

MEMORY = 0.85
pup = Upsampler(MODEL, MEMORY)
print("loaded", MODEL)

## Inputs

Edit and re-run. `IMAGES` are paths (or already loaded `PIL.Image`s); `BOXES` is a list of dicts, a path to a boxes JSON
file, or `None`.

In [ ]:
import json

from IPython.display import display
from PIL import Image

PROMPT = "A man walking with his dog, afternoon sun"
IMAGES: list = []  # e.g. [REPO / "examples/inputs/1.png"]
BOXES = None       # e.g. REPO / "examples/inputs/balloons_boxes.json", or [{"id": "sun", "desc": "...", "bbox": [0.0, 0.6, 0.3, 0.9]}]

images = [p if isinstance(p, Image.Image) else Image.open(p) for p in IMAGES]
boxes = json.loads(Path(BOXES).read_text()) if isinstance(BOXES, (str, Path)) else BOXES

print(f"prompt: {PROMPT!r}\nimages: {len(images)}  boxes: {len(boxes) if boxes else 0}")
for i, image in enumerate(images):
    print(f"ref_image_{i}: {image.size} {image.mode}")
    display(image.copy().reduce(max(1, image.width // 256)))

## Upsample

In [ ]:
caption = pup(PROMPT, images, boxes)
print(caption)

Optional: save the caption, e.g. to feed `scripts/cli.py ... --no-pup` or the RoPE notebook's `PROMPT`.

In [ ]:
# out = REPO / "notebooks" / "outputs" / "caption.txt"
# out.parent.mkdir(exist_ok=True)
# out.write_text(caption + "\n")
# print("saved to", out)